# Market data research and trading strategy evaluation

Florien Siakoua Toukam | October 2026

This notebook presents saved-result tables and code cells for reproducing the analysis. The two experiments are separate: the volatility model does not drive the ETF trades.


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / '03_Code').exists():
    ROOT = ROOT.parent
assert (ROOT / '03_Code/research_plan.json').exists()
sys.path.insert(0, str(ROOT / '03_Code'))
plan = json.loads((ROOT / '03_Code/research_plan.json').read_text())
plan

## 1. Source snapshots and integrity

Six Optiver parquet files cover anonymized stocks 0, 1 and 3. Yahoo snapshots retain SPY, QQQ and IWM prices and corporate actions. The original request URLs, raw byte counts and SHA-256 hashes are recorded in the manifest.

In [ ]:
manifest = json.loads((ROOT / '01_Data/source_manifest.json').read_text())
pd.DataFrame(manifest['files'])[['file','bytes','source','sha256']]

## 2. Five-minute features and prediction target

First-half seconds 0-299 provide all predictors. Second-half seconds 300-599 provide realized volatility to predict. This is a reconstructed five-minute task, not the original Kaggle target. Complete shuffled time buckets stay in one split, but the test is not chronological.

In [ ]:
import analyze_optiver
optiver_summary = analyze_optiver.run()
features = pd.read_csv(ROOT / '01_Data/processed/optiver_five_minute_features.csv')
features.groupby(['stock_id','split']).size().unstack()

### Saved reserved-test results

| model | rmspe | rmse | mae |
| --- | --- | --- | --- |
| Persistence | 34.22% | 0.001478 | 0.000816 |
| Calibrated persistence | 29.69% | 0.001316 | 0.000888 |
| RV-only Ridge | 29.96% | 0.001095 | 0.000718 |
| Ridge | 28.67% | 0.001073 | 0.000693 |
| Gradient boosting | 29.23% | 0.001064 | 0.000698 |

The full Ridge gain is 16.2% versus the naive baseline, 3.4% versus calibration, and 4.3% versus RV-only Ridge. The confidence interval versus calibration includes zero. Forecasting variability is not forecasting direction or trading profit.

In [ ]:
pd.read_csv(ROOT / '04_Results/optiver_model_scores.csv')

![Five-minute volatility model comparison](../05_Charts/optiver_model_comparison.png)

Compare the validation and test results, then inspect the model coefficients. The selected full model is determined by validation RMSPE, not by picking the best test result.

In [ ]:
pd.read_csv(ROOT / '04_Results/optiver_ridge_coefficients.csv')

## 3. Delayed trading-strategy evaluation

The fixed primary rule goes long when SPY closes above its 200-session price average and otherwise holds cash. A signal executes at the next session close and earns returns only thereafter. Costs are 5 bps on one-way traded notional. Main cash interest is zero. Historical later-period evaluation is 2020-September 2026.

### Saved SPY later-period results

| strategy | cagr | annualized_volatility | max_drawdown |
| --- | --- | --- | --- |
| Trend filter | 10.36% | 12.41% | -23.20% |
| Buy and hold | 15.29% | 20.05% | -33.72% |
| 50% ETF / 50% cash | 7.89% | 10.02% | -17.98% |

In [ ]:
import analyze_strategy
strategy_summary = analyze_strategy.run()
pd.read_csv(ROOT / '04_Results/strategy_performance.csv').query("period == 'Held-out'")

![SPY later-period performance](../05_Charts/SPY_held_out_performance.png)

The rule reduces drawdown and volatility but gives up return. The half-stock/half-cash benchmark also shows how lower exposure can reduce risk. The model is not selected by searching the held-out lookbacks.

In [ ]:
pd.read_csv(ROOT / '04_Results/strategy_sensitivity.csv')

## 4. Follow an execution

The saved simulated trade log can show a BUY on a day whose newly observed close signal is CASH. That is consistent with following the previous session decision. Inspect March 2020 whipsaws rather than assuming a delayed trade uses today's information.

In [ ]:
trades = pd.read_csv(ROOT / '04_Results/SPY_simulated_trade_log.csv')
trades.query("date >= '2020-02-01' and date <= '2020-06-30'")

## 5. Independent verification and SQL

Source checksums, raw record counts, non-overlapping splits, future-information perturbation tests, independent portfolio ledgers, and scalar metric recalculations are checked. SQLite provides a separate processed-data audit. See the report for the methodological limitations that remain even when calculation checks pass.

In [ ]:
import verify_project
verification = verify_project.run()
verification['status'], verification['completed_checks']

In [ ]:
import sqlite3
with sqlite3.connect(ROOT / '01_Data/processed/market_research.sqlite') as con:
    summary = pd.read_sql_query('SELECT stock_id, split, COUNT(*) AS n FROM optiver_features GROUP BY stock_id, split', con)
summary

## Conclusion

The project compares short-horizon risk forecasts and a delayed trend strategy. Stronger forecast baselines narrow the measured gain. The trend rule reduces drawdown but gives up return. The memorandum explains the results and their limits.
